## 0 · Configuration & Imports

In [0]:
import requests
import json
import logging
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, ArrayType
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("youtube_ingestion")

spark = SparkSession.builder.appName("youtube_ingestion").getOrCreate()

In [0]:
API_KEY        = dbutils.secrets.get(scope="my-secrets", key="api-key")
print(API_KEY)

[REDACTED]


## 1 · Pipeline Parameters

In [0]:
from datetime import datetime, timezone
dbutils.widgets.text("bucket_name","youtube-capstone-bucket-team6","S3 Bucket Name")
dbutils.widgets.text(
    "region_codes",
    "US,IN,GB,CA,AU",
    "Region Codes (comma separated)"
)
dbutils.widgets.text("max_results","200","Max Results per API call")
dbutils.widgets.text("run_date",datetime.now(timezone.utc).strftime("%Y-%m-%d"), "Run Date")

BUCKET_NAME    = dbutils.widgets.get("bucket_name")

regions = [r.strip().upper() for r in dbutils.widgets.get("region_codes").split(",")]
MAX_RESULTS    = int(dbutils.widgets.get("max_results"))
RUN_DATE       = dbutils.widgets.get("run_date")

# Derived S3 paths
BASE_PATH      = f"s3://{BUCKET_NAME}/youtube_pipeline"
RAW_VIDEOS_PATH   = f"{BASE_PATH}/raw_uploads/raw_videos/run_date={RUN_DATE}"
RAW_COMMENTS_PATH = f"{BASE_PATH}/raw_uploads/raw_comments/run_date={RUN_DATE}"

# YouTube API endpoints
YOUTUBE_TRENDING_URL = "https://www.googleapis.com/youtube/v3/videos"
YOUTUBE_COMMENTS_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

logger.info(f"Ingestion config → region={regions}, max_results={MAX_RESULTS}, run_date={RUN_DATE}")
logger.info(f"Video landing:   {RAW_VIDEOS_PATH}")
logger.info(f"Comment landing: {RAW_COMMENTS_PATH}")
print(regions)
type(regions)

['US', 'IN', 'GB', 'CA', 'AU']


list

## 2 · YouTube API Helpers

In [0]:
def fetch_trending_videos(api_key: str, region_code: str, max_results: int) -> list[dict]:
    
    all_videos = []
    for region_cod in regions:

        region_videos = []
        next_token    = None

        params_base = {
            "part":       "snippet,statistics,contentDetails",
            "chart":      "mostPopular",
            "regionCode": region_cod,
            "maxResults": min(max_results, 200),
            "key":        api_key,
        }

        while len(region_videos) < max_results:
            params = params_base.copy()
            if next_token:
                params["pageToken"] = next_token

            try:
                resp = requests.get(YOUTUBE_TRENDING_URL, params=params, timeout=30)
                resp.raise_for_status()
                payload = resp.json()
                items   = payload.get("items", [])
                for item in items:
                    item["_region_code"] = region_cod
                region_videos += items
                next_token = payload.get("nextPageToken")
                if not next_token:
                    break
            except requests.RequestException as exc:
                logger.error(f"API error fetching videos for {region_cod}: {exc}")
                break

        logger.info(f"Fetched {len(region_videos)} trending videos for region '{region_cod}'")
        all_videos += region_videos[:max_results]

    logger.info(f"Total videos fetched across all regions: {len(all_videos)}")
    return all_videos


def parse_video_record(item: dict, region_code: str, run_date: str) -> dict:
    """Flatten a raw YouTube video resource into a tabular dict."""
    snippet    = item.get("snippet",        {})
    stats      = item.get("statistics",     {})
    content    = item.get("contentDetails", {})
    thumbnails = snippet.get("thumbnails",  {})

    return {
        "video_id":         item.get("id",                           ""),
        "title":            snippet.get("title",                     ""),
        "channel_id":       snippet.get("channelId",                 ""),
        "channel_title":    snippet.get("channelTitle",              ""),
        "publish_date":     snippet.get("publishedAt",               ""),
        "description":      snippet.get("description",               ""),
        "tags":             json.dumps(snippet.get("tags",           [])),
        "category_id":      snippet.get("categoryId",                ""),
        "duration":         content.get("duration",                  ""),
        "thumbnail_url":    thumbnails.get("high", {}).get("url",    ""),
        "view_count":       int(stats.get("viewCount",      0) or 0),
        "like_count":       int(stats.get("likeCount",      0) or 0),
        "comment_count":    int(stats.get("commentCount",   0) or 0),
        "region_code":      region_code,
        "run_date":         run_date,
        "ingested_at":      datetime.now(timezone.utc).isoformat(),
    }


def fetch_comments_for_video(api_key: str, video_id: str, max_comments: int = 100) -> list[dict]:
    """
    Fetch top-level comment threads for a single video.

    Silently skips videos with comments disabled (HTTP 403 commentDisabled).
    """
    comments   = []
    next_token = None

    params_base = {
        "part":       "snippet",
        "videoId":    video_id,
        "maxResults": min(max_comments, 100),
        "order":      "relevance",
        "key":        api_key,
    }

    while len(comments) < max_comments:
        params = params_base.copy()
        if next_token:
            params["pageToken"] = next_token

        try:
            resp = requests.get(YOUTUBE_COMMENTS_URL, params=params, timeout=30)
            if resp.status_code == 403:
                logger.warning(f"Comments disabled for video {video_id} – skipping")
                break
            resp.raise_for_status()
            payload    = resp.json()
            comments  += payload.get("items", [])
            next_token = payload.get("nextPageToken")
            if not next_token:
                break
        except requests.RequestException as exc:
            logger.error(f"Error fetching comments for {video_id}: {exc}")
            break

    return comments[:max_comments]


def parse_comment_record(item: dict, video_id: str, run_date: str) -> dict:
    """Flatten a raw comment thread resource into a tabular dict."""
    top = item.get("snippet", {}).get("topLevelComment", {}).get("snippet", {})

    return {
        "comment_id":   item.get("id",                              ""),
        "video_id":     video_id,
        "author":       top.get("authorDisplayName",                ""),
        "comment_text": top.get("textDisplay",                      ""),
        "like_count":   int(top.get("likeCount",  0) or 0),
        "comment_date": top.get("publishedAt",                      ""),
        "run_date":     run_date,
        "ingested_at":  datetime.now(timezone.utc).isoformat(),
    }

## 3 · Ingest Videos

In [0]:
logger.info("── Starting video ingestion ─-")

raw_video_items = fetch_trending_videos(API_KEY, regions, MAX_RESULTS)
video_records   = [parse_video_record(item, item.get("_region_code", ""), RUN_DATE) for item in raw_video_items]

if not video_records:
    raise ValueError("No video records returned. Check API key, quota, and region code.")

df_videos = spark.createDataFrame(video_records)
df_videos = df_videos.withColumn("_pipeline_ts", current_timestamp())

(
    df_videos
    .write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .parquet(RAW_VIDEOS_PATH)
)

logger.info(f"Saved {df_videos.count()} video records → {RAW_VIDEOS_PATH}")
df_videos.printSchema()

root
 |-- category_id: string (nullable = true)
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- description: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- like_count: long (nullable = true)
 |-- publish_date: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- run_date: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- thumbnail_url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- view_count: long (nullable = true)
 |-- _pipeline_ts: timestamp (nullable = false)



## 4 · Ingest Comments

In [0]:
logger.info("── Starting comment ingestion ──")

video_ids = [r["video_id"] for r in video_records]
comment_records = []

for vid_id in video_ids:
    raw_comments     = fetch_comments_for_video(API_KEY, vid_id, max_comments=50)
    parsed           = [parse_comment_record(c, vid_id, RUN_DATE) for c in raw_comments]
    comment_records += parsed

logger.info(f"Total comments collected: {len(comment_records)}")

if comment_records:
    df_comments = spark.createDataFrame(comment_records)
    df_comments = df_comments.withColumn("_pipeline_ts", current_timestamp())

    (
        df_comments
        .write
        .mode("overwrite")
        .option("mergeSchema", "true")
        .parquet(RAW_COMMENTS_PATH)
    )
    logger.info(f"Saved {df_comments.count()} comment records → {RAW_COMMENTS_PATH}")
    df_comments.printSchema()
else:
    logger.warning("No comments collected. Writing empty sentinel file.")
    spark.createDataFrame([], schema="comment_id STRING, video_id STRING") \
         .write.mode("overwrite").parquet(RAW_COMMENTS_PATH)

root
 |-- author: string (nullable = true)
 |-- comment_date: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- like_count: long (nullable = true)
 |-- run_date: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- _pipeline_ts: timestamp (nullable = false)



## 5 · Validation & Summary

In [0]:
# Quick row-count validation
v_count = spark.read.parquet(RAW_VIDEOS_PATH).count()
c_count = spark.read.parquet(RAW_COMMENTS_PATH).count()

print("=" * 60)
print("  INGESTION COMPLETE — SUMMARY")
print("=" * 60)
print(f"  Run Date       : {RUN_DATE}")
print(f"  Region         : {','.join(regions)}")
print(f"  Videos landed  : {v_count:,}")
print(f"  Comments landed: {c_count:,}")
print(f"  Videos path    : {RAW_VIDEOS_PATH}")
print(f"  Comments path  : {RAW_COMMENTS_PATH}")
print("=" * 60)

dbutils.notebook.exit(json.dumps({
    "status":         "success",
    "run_date":       RUN_DATE,
    "videos_count":   v_count,
    "comments_count": c_count,
}))

In [0]:
print(v_count)


999


In [0]:
print(c_count)

38026


In [0]:
display(df_videos)

category_id channel_id channel_title comment_count description duration ingested_at like_count publish_date region_code run_date tags thumbnail_url title video_id view_count _pipeline_ts 10 UCm1hg6Vg2gr5Od8SBPRxGlw Hey! 6 🌹((Artista))
@romeo 

📸 ((Letra))
Hey, the king is back
Ajá
No te imaginas el placer y honor que es tenerte como amigo
Cuántos años recorridos
Yo soy tu hermano y un soldado en devoción
Los golpes en la vida son muy crueles y a veces sale herido
Aquel que no lo ha merecido
Anoche fui testigo de una traición
Vi a tu esposa con un hombre cariñosa (Ah-ah-ah)
Los primos no se besan en la boca (Ah-ah-ah)
Lo que te ha hecho es un delito
Y sus besos son malditos, falso amor
La vi bien sospechosa cuando entraba a un motel
Con gafas y peluca, el traje rojo de Channel
El que le diste pa' su aniversario, la muy descarada
En tu propio BM, lo montó y con el se fue
Y le tiré dos fotos para prueba que es infiel
Mientras le das el cielo, ella con otro se estruja en la cama
Lo que te ha hecho es un delito
Y sus besos son malditos, falso amor
Gustoso
Hay amigo mío
No sea palomo (She gotta do it again)
Es una bárbara
Vi a tu esposa con un hombre cariñosa (Ah-ah-ah)
Y los primos no se besan en la boca (Ah-ah-ah)
Lo que te ha hecho es un delito
Y sus besos son malditos, falso amor
La vi bien sospechosa cuando entraba a un motel
Con gafas y peluca el traje rojo de Channel
El que le diste pa' su aniversario la muy descarada
En tu propio BM lo montó y con el se fue
Y le tiré dos fotos para prueba que es infiel
Mientras le das el cielo, ella con otro se estruja en la cama
Lo que te ha hecho es un delito
Y sus besos son malditos, falso amor
Abran paso
Hay mis hijos
Ella te engaña no comprende el concepto de ser dama
(Falso amor)
Amigo mío no seas tonto, no hagas caso a sus palabras
(Falso amor)
(Listen to me)
Arrogando con sus besos, sus caricias y su cuerpo que malvada
(Falso amor)
Le diste la luna, una estrella y mira como paga
(Falso amor)
Te ha jugado una sucieza no merece tu perdón

🌱 ((Tags ))
letra, español, traducido al español. lyrics, letra español, traducción
música, mix, canciones mix, indie, alternativo, canciones tristes, vibes, musica 2025, canciones de rock, canciones rock viejitas en ingles, canciones alternativas, canciones de amor, canciones de desamor, canciones para llorar mucho y asi, canciones indie, rock mix music
canciones playlist, canciones para llorar, canciones tranquilas, canciones con traducción, canciones de rock, canciones pop, indie lyric videos, alternative rock lyrics.
Romeo Santos - Amigo ;español
Romeo Santos - Amigo ;traducción español
Romeo Santos - Amigo ;lyrics
Romeo Santos - Amigo ;mix playlist

#alternativo #rock #indie #alternativo #pop #rockpop #oldmusic PT3M50S 2026-04-27T08:11:42.570768+00:00 253 2026-04-23T18:15:06Z US 2026-04-19 ["letra", "traducido al espa\u00f1ol. lyrics", "letra espa\u00f1ol", "traducci\u00f3n m\u00fasica", "mix", "canciones mix", "indie", "canciones tristes", "canciones de amor", "canciones de desamor", "canciones para llorar mucho y asi", "canciones indie", "rock mix music canciones playlist", "canciones para llorar", "canciones tranquilas", "canciones con traducci\u00f3n", "canciones pop", "hey lyrics", "hey!", "musica romantica para dedicar", "romeo santos amigo letra", "romeo amigo letra", "amigo romeo santos letra", "romeo santos amigo", "romeo sant"] https://i.ytimg.com/vi/tSp_BJ5UaUc/hqdefault.jpg Romeo Santos - Amigo (Letra) tSp_BJ5UaUc 248435 2026-04-27T08:15:39.769Z 20 UCKy1dAqELo0zrOtPkf0eTMw IGN 1237 A feeling of being safer than one really is...
Take a look at the teaser for the long-awaited sequel to Alien: Isolation, the survival horror game developed by Creative Assembly.

#ign #gaming PT26S 2026-04-27T08:11:42.570795+00:00 11390 2026-04-26T18:58:06Z US 2026-04-19 ["Action", "Adventure", "Alien", "Alien: Isolation", "Android", "Creative Assembly", "Feral Interactive", "Nintendo Switch", "PC", "PlayStation 3", "PlayStation 4", "SEGA", "Steam Deck Ver